In [1]:
#Routr project

#The goal is to design an app that plans a route through Madison/Boulder for you based off of bike lanes using Dijkstra's
#Or a Euler trail for certain destinations for sightseeing.

In [2]:
import os
import geopandas
import osmnx as ox
import networkx as nx
import numpy as np
import heapq as hq
import streamlit as st
import folium 

import streamlit_folium as sf


2025-11-07 11:48:09.895 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [ ]:
#How the overall app should work:

#First thing a search bar with your desired city:
    #Enter city, state

#Ask someone about how they want to be routed:
    #Quickest, Safest, Bike-focused
    #Tool-tip on each about how each option is structured.

#Next pull up a folium explore location of the city:
    # Don't use on the gdf, will be too slow loading in the geometries
    #Provide a click operation where someone selects their start & end points
    #first click is assigned to class.start_node
    #second click is assigned to class.end_node

#Build class Once city, preference, + start&end_nodes are provided:
    #Automatically call the shortest_path queue function
    #Return total route distance
    #Use shortest_path to create a route_gdf 
    #Use the exlore() function/


In [ ]:
class CyclewayGraph:
    
    def __init__(self, city, state, route_pref = 'bike-focused'):
        self.id = f"{city}, {state}"

        self.city = city
        self.state = state

        self.route_pref = route_pref

        self.graph = ox.graph_from_place(query = self.id, network_type= 'bike')

        self.nodes = self.graph.nodes

        # cycleway_edges_unsort = self.bike_gdf_edges.loc[self.bike_gdf_edges.loc[:,'highway']=='cycleway']

        # self.graph = bike_data

        

    def create_path_list(self, start_node, end_node, previous_nodes, node_distances, path_distance = False):
        """Uses the get_shortest_path function's inputs & outputs of previous_nodes & node_distances to return a list of the nodes to be traversed.
            Has the option of including the distances for each edge to traverse if specified.  

        Args:
            start_node (int): the beginning node to navigate from.
            end_node (int): the end node destination desired.
            previous_nodes (dict): a dictionary of nodes and the node to travel from along the shorest path to each node.
            node_distances (dict): a dictionary of each node and the shortest distance required to reach each node.
            path_distance (bool, optional): if True will provide a separate list the distance of each edge traversed. Defaults to False.

        Returns:
            list: a list of shortest path of nodes that should be traversed, in order from start to finish.
        """

        shortest_path = list()
        shortest_path_dists = list()
        closest_node = end_node

        if path_distance == True:
            total_path_length = node_distances[closest_node]
            while previous_nodes[closest_node]:
                shortest_path.append(closest_node)
                
                edge_end_dist = node_distances[closest_node] #cumulative distance along the path to the END of the edge
                edge_start_dist = node_distances[previous_nodes[closest_node]] #cumulative distance along the path to the START of the edge, gathered once reassigned
                current_edge_length = edge_end_dist - edge_start_dist
                shortest_path_dists.append(current_edge_length)
                if previous_nodes[closest_node] == start_node:
                    shortest_path.append(start_node)
                    return shortest_path[::-1], shortest_path_dists[::-1]
            
                closest_node = previous_nodes[closest_node]
                        
        else:
            while previous_nodes[closest_node]:
                shortest_path.append(closest_node)
                if previous_nodes[closest_node] == start_node:
                    shortest_path.append(start_node)
                    return shortest_path[::-1]
                
                closest_node = previous_nodes[closest_node]

    def edge_calculator(self, edge_data, preference):
        """Uses nested if statements with multipliers to weight the user's route towards their path preferences  

        Args:
            edge_data (dict): The edge dictionary accessed by indexing the edge's start node, end node, and edge index to the graph.
            preference (str): One of the available routing preferences selectable ("safest", "bike-focused", or "quickest").

        Returns:
            np.float64: the weighted edge length based off of the user's preferences.
        """ 
        
        highway_type = edge_data['highway']
        edge_length = edge_data['length']

        #Would converting the lists to sets be faster? probably unnecessary...

        if preference == 'safest':
            if 'cycleway' in highway_type or 'path' in highway_type:
                weighted_length = edge_length * 0.7
                return weighted_length
            
            elif ('residential' in highway_type) or ('service' in highway_type) or (('track') in highway_type):
                weighted_length = edge_length * 0.85
                return weighted_length
            
            else:
                weighted_length = edge_length * 1.2
                return weighted_length
            
        elif preference == 'bike-focused':
            if 'cycleway' in highway_type or 'path' in highway_type:
                weighted_length = edge_length * 0.8
                return weighted_length
            
            elif ('residential' in highway_type) or ('service' in highway_type) or (('track') in highway_type):
                weighted_length = edge_length * 0.9
                return weighted_length
            
            else:
                weighted_length = edge_length * 1.1
                return weighted_length
            
        else:
            # shortest/default option
            return edge_length


    def find_shortest_path_queue(self, start_node, end_node, path_distance = False, override_pref = False):
        """This function implements a priority queue, using heapq, to find the shortest path between two nodes in a NetworkX MultiDiGraph.

        Args:
            start_node (int): the beginning node to navigate from.
            end_node (int): the end node destination desired.
            path_distance (bool, optional): If True will provide a separate list the distance of each edge traversed. Defaults to False.

        Returns:
            list: a list of shortest path of nodes that should be traversed, in order from start to finish.
        """
        
        node_distances = {node : float('inf') for node in list(self.nodes)}
        node_distances[start_node] = 0
        previous_nodes = dict()
        shortest_path = list()
        shortest_path_dists = list() #length will be different, because it returns the distance per edge, while shortest_path is a list of all nodes

        unvisited_nodes = [(value, node) for node, value in node_distances.items()]
        hq.heapify(unvisited_nodes)

        while len(unvisited_nodes) != 0:
            closest_node_dist, closest_node = hq.heappop(unvisited_nodes)

            if closest_node == end_node:
                return self.create_path_list(start_node, end_node, previous_nodes, node_distances, path_distance)

            for neighbor_node in self.graph[closest_node]: #This index provides the adjacent nodes to the current closest node

                edge_option = min(self.graph[closest_node][neighbor_node], key = lambda edge_id: self.graph[closest_node][neighbor_node].get(edge_id)['length'] )
                
                edge_data = self.graph[closest_node][neighbor_node][edge_option]

                # edge_length = self.graph[closest_node][neighbor_node][edge_option]['length'] #NEED TO LOOK AT EACH EDGE WITHIN THE PREVIOUS NODES, currently assuming 0
                if override_pref:
                    edge_length = self.edge_calculator(edge_data, preference = override_pref) 
                else:
                    edge_length = self.edge_calculator(edge_data, preference = self.route_pref)
                
                alt_path_length = closest_node_dist + edge_length

                if node_distances[neighbor_node] == alt_path_length:
                    continue

                if node_distances[neighbor_node] > alt_path_length:
                    node_distances[neighbor_node] = alt_path_length
                    previous_nodes[neighbor_node] = closest_node
                    hq.heappush(unvisited_nodes, (alt_path_length, neighbor_node))

        

In [4]:
# start_node = 11143855834
# end_node = 1179967122

start_node = 1179967122
end_node = 11143855834

In [5]:
msn_bike_graph = CyclewayGraph('Madison','Wisconsin')

In [6]:
shortest_path, path_dists = msn_bike_graph.find_shortest_path_queue(start_node, end_node, path_distance = True, override_pref = 'quickest')

In [7]:
# safest_route_gdf = ox.routing.route_to_gdf(msn_bike_graph.graph, safest_path, weight='length')

In [9]:
def route_mapper(shortest_path):
    # https://getbootstrap.com/docs/3.3/components/

    route_gdf = ox.routing.route_to_gdf(msn_bike_graph.graph, shortest_path, weight='length')
    route_map = route_gdf.explore()

    folium.Marker(
        location=[msn_bike_graph.graph.nodes[start_node]['y'], msn_bike_graph.graph.nodes[start_node]['x']],
        tooltip="Route Start",
        popup="Starting Point",
        icon=folium.Icon(icon= "glyphicon-user", color = "green"),
    ).add_to(route_map)

    folium.Marker(
        location=[msn_bike_graph.graph.nodes[end_node]['y'], msn_bike_graph.graph.nodes[end_node]['x']],
        tooltip="End Destination",
        popup="End Destination",
        icon=folium.Icon(icon= "glyphicon-remove", color = "red"),
    ).add_to(route_map)

    return route_map

In [10]:
route_mapper(shortest_path)

In [ ]:
# safest_route_map = ox.routing.route_to_gdf(msn_bike_graph.graph, safest_path, weight='length').explore()

In [ ]:
folium.Map(location=[45.523, -122.675], width=750, height=500)

In [ ]:
#Keep in mind that some cycleway highways have 3 or more labels ['cycleway', 'path', 'residential']

#all types in the network: 
# {'cycleway', 'service', 'unclassified', 'track', 'trunk', 'busway', 'tertiary', 'pedestrian', 'secondary_link', 'path', 'residential', 'secondary', 'tertiary_link', 'primary', 'trunk_link', 'primary_link'}


#Mapping out edge_calculator:
    # quickest:
        # Just the shortest path

    # safest:
        # heavily favor cycleway, path, pedestrian??  0.8x?
        # slightly favor residential, service, track? 0.9x?
        # penalize motorway, trunk, primary, secondary, & all links 1.2x?

    #bike - focused:
        # favor cycleway, path, pedestrian?? 0.9x
        # slightly favor residential, service?
        # penalize motorway, trunk, primary, secondary, & all links 1.1x?

In [ ]:
def find_shorest_neighbor_lengths(start_node):
    """Iterates over the neighboring nodes of the input start_node to find their edge lengths.

    Args:
        start_node (int): the beginning node id 
    """
    
    neighbor_nodes = list(bike_graph.neighbors(start_node))

    for neighbor in neighbor_nodes:
        neighbor_edges_dict = bike_graph[start_node][neighbor]
        for edge_key in neighbor_edges_dict:
            edge = neighbor_edges_dict[edge_key]
            edge_length = edge['length']
            return edge_length

In [ ]:
def find_shorest_neighbor_edge(start_node):
    """Iterates over the neighboring nodes of the input start_node to find their edge lengths.

    Args:
        start_node (int): the beginning node id 
    """
    
    neighbor_nodes = list(bike_graph.neighbors(start_node))

    for neighbor in neighbor_nodes:
        neighbor_edges_dict = bike_graph[start_node][neighbor]
        closest_neighbor_node = min(neighbor_edges_dict, key = neighbor_edges_dict.get)
        return closest_neighbor_node

In [ ]:
def find_shortest_path(start_node, end_node):
    """Implements Dijkstra's algorithm to find the shortest cycleway path between two nodes in a NetworkX MultiDiGraph.

    Args:
        start_node (int): the beginning node to navigate from.
        end_node (int): the end node destination desired.

    Returns:
        list: a list of shortest path of nodes that should be traversed, in order from start to finish.
    """
    node_distances = {node : float('inf') for node in list(bike_graph.nodes)}
    node_distances[start_node] = 0
    previous_nodes = dict()
    unvisited_nodes = set(bike_graph.nodes)
    shortest_path = list()

    while len(unvisited_nodes) != 0:
        min_distance_so_far = float("inf")
        closest_node = None
        for node, dist in node_distances.items():
            if ((node in unvisited_nodes) & (dist < min_distance_so_far)):
                closest_node = node
                min_distance_so_far = dist

        if closest_node == end_node:
            while previous_nodes[closest_node]:
                shortest_path.append(closest_node)
                if previous_nodes[closest_node] == start_node:
                    shortest_path.append(start_node)
                    return shortest_path[::-1]
                closest_node = previous_nodes[closest_node]
            
                #Something is wrong here...

        if closest_node == None:
            break
        #Case for disconnected graph, if the nodes aren't connected, then the
        #distance of the disconnected nodes will never be updated to not infinity

            

        unvisited_nodes.remove(closest_node)
        
        for neighbor_node in bike_graph[closest_node]: #This indexing provides the adjacent nodes to the current closest node
            if neighbor_node in unvisited_nodes:
                edge_option = min(bike_graph[closest_node][neighbor_node], key = lambda edge_id: bike_graph[closest_node][neighbor_node].get(edge_id)['length'] )
                edge_length = bike_graph[closest_node][neighbor_node][edge_option]['length'] #NEED TO LOOK AT EACH EDGE WITHIN THE PREVIOUS NODES, currently assuming 0
                alt_path_length = node_distances[closest_node] + edge_length
                if node_distances[neighbor_node] > alt_path_length:
                    node_distances[neighbor_node] = alt_path_length
                    previous_nodes[neighbor_node] = closest_node

            else:
                continue


    

In [ ]:
def create_path_list(start_node, end_node, previous_nodes, node_distances, path_distance = False):
    """Uses the get_shortest_path function's inputs & outputs of previous_nodes & node_distances to return a list of the nodes to be traversed.
        Has the option of including the distances for each edge to traverse if specified.  

    Args:
        start_node (int): the beginning node to navigate from.
        end_node (int): the end node destination desired.
        previous_nodes (dict): a dictionary of nodes and the node to travel from along the shorest path to each node.
        node_distances (dict): a dictionary of each node and the shortest distance required to reach each node.
        path_distance (bool, optional): if True will provide a separate list the distance of each edge traversed. Defaults to False.

    Returns:
        list: a list of shortest path of nodes that should be traversed, in order from start to finish.
    """
    shortest_path = list()
    shortest_path_dists = list()
    closest_node = end_node

    if path_distance == True:
        total_path_length = node_distances[closest_node]
        while previous_nodes[closest_node]:
            shortest_path.append(closest_node)
            
            edge_end_dist = node_distances[closest_node] #cumulative distance along the path to the END of the edge
            edge_start_dist = node_distances[previous_nodes[closest_node]] #cumulative distance along the path to the START of the edge, gathered once reassigned
            current_edge_length = edge_end_dist - edge_start_dist
            shortest_path_dists.append(current_edge_length)
            if previous_nodes[closest_node] == start_node:
                shortest_path.append(start_node)
                return shortest_path[::-1], shortest_path_dists[::-1]
        
            closest_node = previous_nodes[closest_node]
                    
    else:
        while previous_nodes[closest_node]:
            shortest_path.append(closest_node)
            if previous_nodes[closest_node] == start_node:
                shortest_path.append(start_node)
                return shortest_path[::-1]
            
            closest_node = previous_nodes[closest_node]

In [ ]:
def find_shortest_path_queue(start_node, end_node, path_distance = False):
    """This function implements a priority queue, using heapq, to find the shortest path between two nodes in a NetworkX MultiDiGraph.

    Args:
        start_node (int): the beginning node to navigate from.
        end_node (int): the end node destination desired.
        path_distance (bool, optional): If True will provide a separate list the distance of each edge traversed. Defaults to False.

    Returns:
        list: a list of shortest path of nodes that should be traversed, in order from start to finish.
    """
    
    node_distances = {node : float('inf') for node in list(bike_graph.nodes)}
    node_distances[start_node] = 0
    previous_nodes = dict()
    shortest_path = list()
    shortest_path_dists = list() #length will be different, because it returns the distance per edge, while shortest_path is a list of all nodes

    unvisited_nodes = [(value, node) for node, value in node_distances.items()]
    hq.heapify(unvisited_nodes)

    while len(unvisited_nodes) != 0:
        closest_node_dist, closest_node = hq.heappop(unvisited_nodes)

        if closest_node == end_node:
            return create_path_list(start_node, end_node, previous_nodes, node_distances, path_distance)

        for neighbor_node in bike_graph[closest_node]: #This index provides the adjacent nodes to the current closest node

            edge_option = min(bike_graph[closest_node][neighbor_node], key = lambda edge_id: bike_graph[closest_node][neighbor_node].get(edge_id)['length'] )
            edge_length = bike_graph[closest_node][neighbor_node][edge_option]['length'] #NEED TO LOOK AT EACH EDGE WITHIN THE PREVIOUS NODES, currently assuming 0
            alt_path_length = closest_node_dist + edge_length

            if node_distances[neighbor_node] == alt_path_length:
                continue

            if node_distances[neighbor_node] > alt_path_length:
                node_distances[neighbor_node] = alt_path_length
                previous_nodes[neighbor_node] = closest_node
                hq.heappush(unvisited_nodes, (alt_path_length, neighbor_node))
        

In [ ]:
class CyclewayGraph_gdf:
    
    def __init__(self, city, state):
        self.id = f"{city}, {state}"

        self.city = city
        self.state = state

        bike_data = ox.graph_from_place(query = self.id, network_type= 'bike')

        self.bike_gdf_nodes, self.bike_gdf_edges = ox.graph_to_gdfs(bike_data)

        cycleway_edges_unsort = self.bike_gdf_edges.loc[self.bike_gdf_edges.loc[:,'highway']=='cycleway']

        self.cyc_edges = cycleway_edges_unsort.sort_index()

        self.all_nodes = set(self.bike_gdf_nodes.index)

    def shortest_path(self, start, end):

        unvisited_nodes = {node_id : float('inf') for node_id in self.all_nodes} #gather all nodes in a dictionary 
        unvisited_nodes[start] = 0 #set starting node length to 0

        prev_nodes = {node_id : np.nan for node_id in self.all_nodes}

        current_node = start #Should be unnecessary

        while len(unvisited_nodes) != 0:
            
            shortest_next_edge_length =  self.cyc_edges.loc[current_node,'length'].min() #somethings wrong here.
            
            minimum_dist_node = min(unvisited_nodes, key = unvisited_nodes.get)

            removed_len  = unvisited_nodes.pop(minimum_dist_node)
            for next_node in self.cyc_edges.loc[current_node].index.get_level_values('v'):
                new_length = removed_len + self.cyc_edges.loc[(13107812002,13107783597), 'length'].item()
                if new_length < unvisited_nodes[next_node]:
                    unvisited_nodes[next_node] = new_length
                    prev_nodes[next_node] = current_node

            return unvisited_nodes



In [ ]:
bike_network = msn_bike.bike_gdf_edges.explore()